In [1]:
# Imports Libraries
from itertools import count
import random
import heapq
import copy
import time

In [2]:
# Puzzle Class
class Puzzle:
    def __init__(self, size, board=None):
        self.size = size  # 3 or 4
        if board:
            self.board = board
        else:
            self.board = [i for i in range(size * size)]

    def find_zero(self):
        idx = self.board.index(0)
        return divmod(idx, self.size)

    def is_goal(self):
      return self.board == [i for i in range(1, self.size * self.size)] + [0]

    def possible_moves(self):
        moves = []
        x, y = self.find_zero()
        directions = {'Up': (x-1, y), 'Down': (x+1, y), 'Left': (x, y-1), 'Right': (x, y+1)}

        for move, (nx, ny) in directions.items():
            if 0 <= nx < self.size and 0 <= ny < self.size:
                new_board = self.board.copy()
                zero_idx = x * self.size + y
                swap_idx = nx * self.size + ny
                new_board[zero_idx], new_board[swap_idx] = new_board[swap_idx], new_board[zero_idx]
                moves.append((move, Puzzle(self.size, new_board)))
        return moves

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(tuple(self.board))

    def __str__(self):
        s = ""
        for i in range(self.size):
            s += ' '.join(f"{n:2}" for n in self.board[i*self.size:(i+1)*self.size]) + "\n"
        return s

In [3]:
# Random Walk Generator
def random_walk(start_puzzle, steps):
    current = start_puzzle
    for _ in range(steps):
        moves = current.possible_moves()
        move, next_puzzle = random.choice(moves)
        current = next_puzzle
    return current

In [4]:
# Heuristics
def misplaced_tiles(puzzle):
    return sum(1 for i, tile in enumerate(puzzle.board) if tile != 0 and tile != i)

def manhattan_distance(puzzle):
    dist = 0
    for idx, tile in enumerate(puzzle.board):
        if tile == 0:
            continue
        goal_x, goal_y = divmod(tile, puzzle.size)
        curr_x, curr_y = divmod(idx, puzzle.size)
        dist += abs(goal_x - curr_x) + abs(goal_y - curr_y)
    return dist

In [5]:
# Search Algorithms
def bfs(start):
    frontier = [(start, [])]
    explored = set()
    nodes_expanded = 0

    while frontier:
        current, path = frontier.pop(0)
        if current.is_goal():
            return path, nodes_expanded
        explored.add(current)
        for move, neighbor in current.possible_moves():
            if neighbor not in explored:
                frontier.append((neighbor, path + [move]))
                explored.add(neighbor)
        nodes_expanded += 1
    return None, nodes_expanded

def astar(start, heuristic):
    frontier = []
    counter = count()  # Unique sequence count
    heapq.heappush(frontier, (heuristic(start), 0, next(counter), start, []))
    explored = set()
    nodes_expanded = 0

    while frontier:
        estimated_total, cost_so_far, _, current, path = heapq.heappop(frontier)
        if current.is_goal():
            return path, nodes_expanded
        explored.add(current)
        for move, neighbor in current.possible_moves():
            if neighbor not in explored:
                g = cost_so_far + 1
                f = g + heuristic(neighbor)
                heapq.heappush(frontier, (f, g, next(counter), neighbor, path + [move]))
                explored.add(neighbor)
        nodes_expanded += 1
    return None, nodes_expanded

In [6]:
# Experiment Function
def run_experiments(size, steps_list):
    initial = Puzzle(size)
    results = []

    for steps in steps_list:
        for i in range(3):  # Generate 3 problems per step size
            random_puzzle = random_walk(initial, steps)
            print(f"\nProblem (Size {size}x{size}) Random Walk {steps} Steps:")

            # Display the start state as a grid
            for row in range(size):
                print(random_puzzle.board[row*size:(row+1)*size])

            # --- Breadth-First Search ---
            start_time = time.time()
            path_bfs, nodes_bfs = bfs(random_puzzle)
            bfs_time = time.time() - start_time
            print("\nBFS:")
            print(f"Solution: {path_bfs}")
            print(f"Length of solution: {len(path_bfs) if path_bfs else 'N/A'}")
            print(f"Nodes expanded: {nodes_bfs}")

            # --- A* with Misplaced Tiles ---
            start_time = time.time()
            path_astar_misplaced, nodes_astar_misplaced = astar(random_puzzle, misplaced_tiles)
            misplaced_time = time.time() - start_time
            print("\nA* with Misplaced Tiles:")
            print(f"Solution: {path_astar_misplaced}")
            print(f"Length of solution: {len(path_astar_misplaced) if path_astar_misplaced else 'N/A'}")
            print(f"Nodes expanded: {nodes_astar_misplaced}")

            # --- A* with Manhattan Distance ---
            start_time = time.time()
            path_astar_manhattan, nodes_astar_manhattan = astar(random_puzzle, manhattan_distance)
            manhattan_time = time.time() - start_time
            print("\nA* with Manhattan Distance:")
            print(f"Solution: {path_astar_manhattan}")
            print(f"Length of solution: {len(path_astar_manhattan) if path_astar_manhattan else 'N/A'}")
            print(f"Nodes expanded: {nodes_astar_manhattan}")

            # Collecting results
            results.append({
                'start_state': random_puzzle.board,
                'steps_walked': steps,
                'bfs': {
                    'full_solution': path_bfs,
                    'path_len': len(path_bfs) if path_bfs else None,
                    'nodes_expanded': nodes_bfs,
                    'time': bfs_time,
                },
                'astar_misplaced': {
                    'full_solution': path_astar_misplaced,
                    'path_len': len(path_astar_misplaced) if path_astar_misplaced else None,
                    'nodes_expanded': nodes_astar_misplaced,
                    'time': misplaced_time,
                },
                'astar_manhattan': {
                    'full_solution': path_astar_manhattan,
                    'path_len': len(path_astar_manhattan) if path_astar_manhattan else None,
                    'nodes_expanded': nodes_astar_manhattan,
                    'time': manhattan_time,
                }
            })
    return results

In [8]:
# Inputs
steps_list = [5, 10, 20, 40, 80]

# 3x3 puzzles
results_3x3 = run_experiments(3, steps_list)


Problem (Size 3x3) Random Walk 5 Steps:
[1, 0, 4]
[3, 5, 2]
[6, 7, 8]

BFS:
Solution: ['Down', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Left', 'Up', 'Right', 'Down', 'Left', 'Down', 'Right', 'Up', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Down', 'Right']
Length of solution: 23
Nodes expanded: 108012

A* with Misplaced Tiles:
Solution: ['Down', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Left', 'Up', 'Right', 'Down', 'Left', 'Down', 'Right', 'Up', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Down', 'Right']
Length of solution: 23
Nodes expanded: 117240

A* with Manhattan Distance:
Solution: ['Down', 'Left', 'Down', 'Right', 'Up', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Up', 'Left', 'Down', 'Right', 'Down', 'Right', 'Up', 'Left', 'Up', 'Right', 'Down', 'Down']
Length of solution: 23
Nodes expanded: 82112

Problem (Size 3x3) Random Walk 5 Steps:
[3, 1, 2]
[0, 4, 5]
[6, 7, 8]

BFS:
Solution: ['Right', 'Right', 'Up', 'Left', 'Left', 'Down', 'Down', 'Right',

In [ ]:
# 4x4 puzzles
results_4x4 = run_experiments(4, steps_list)